# Respuesta sugerida  Unidad-03

In [1]:
# Importación de las librerías necesarias
import pandas as pd
import numpy as np

In [2]:
# Crear dataset inicial
df = pd.DataFrame({
    'id_producto': [101, 102, 103, 103, 104, 105, 106, 107, 108, 109, 110, 111],
    'nombre': [' Laptop HP  ', 'Mouse Logitech', 'Teclado Mecanico', 'Teclado Mecanico', 
               'Monitor Samsung', None, 'Laptop Dell', 'Auriculares Sony', 
               'Tablet Samsung', 'LAPTOP LENOVO', 'Laptop lenovo', 'Webcam Logitech'],
    'categoria': ['Laptop', 'accesorios', 'ACCESORIOS', 'ACCESORIOS', 'Monitores', 
                  'laptop', 'Laptop', 'Accesorios', 'Tablets', 'laptop', 'Laptops', 'accesorios'],
    'precio': [1200.50, 25.00, 89.90, 89.90, -50.00, 1500.00, 1350.00, 
               45.50, 650.00, 1100.00, 1100.00, 55.00],
    'stock': [15, 120, 45, 45, -10, 8, 22, 80, 35, 12, 12, 95],
    'peso_kg': [2.5, 0.15, 0.8, 0.8, None, None, 2.3, 0.25, 0.55, 2.4, 2.4, 0.18],
    'calificacion': [4.5, 4.2, None, None, 3.8, None, 4.7, 4.0, 4.3, None, None, 3.9],
    'fecha_ingreso': ['2024-05-15', '15/06/2024', '2024-07-20', '2024-07-20', 
                      '08-10-2024', None, '2024-09-12', '20/10/2024', 
                      '2024-11-05', '11-15-2024', '11-15-2024', '2024-12-01'],
    'pais_origen': ['AR', 'CL', 'CO', 'CO', 'AR', 'CL', 'AR', 'CL', 'CO', 'AR', 'AR', 'CL']
})

In [3]:
df

,id_producto,nombre,categoria,precio,stock,peso_kg,calificacion,fecha_ingreso,pais_origen
0,101,Laptop HP,Laptop,1200.5,15,2.50,4.5,2024-05-15,AR
1,102,Mouse Logitech,accesorios,25.0,120,0.15,4.2,15/06/2024,CL
2,103,Teclado Mecanico,ACCESORIOS,89.9,45,0.80,NaN,2024-07-20,CO
3,103,Teclado Mecanico,ACCESORIOS,89.9,45,0.80,NaN,2024-07-20,CO
4,104,Monitor Samsung,Monitores,-50.0,-10,NaN,3.8,08-10-2024,AR
5,105,None,laptop,1500.0,8,NaN,NaN,None,CL
6,106,Laptop Dell,Laptop,1350.0,22,2.30,4.7,2024-09-12,AR
7,107,Auriculares Sony,Accesorios,45.5,80,0.25,4.0,20/10/2024,CL
8,108,Tablet Samsung,Tablets,650.0,35,0.55,4.3,2024-11-05,CO
9,109,LAPTOP LENOVO,laptop,1100.0,12,2.40,NaN,11-15-2024,AR


In [4]:
# Métricas iniciales
registros_iniciales = len(df)
print(f"1. DATASET INICIAL: {registros_iniciales} registros")

1. DATASET INICIAL: 12 registros


In [5]:
# PASO 1: DETECCIÓN DE PROBLEMAS
print("2. DETECCIÓN DE PROBLEMAS:")
print(f"   - Faltantes por columna:\n{df.isna().sum()}")
print(f"   - Duplicados exactos: {df.duplicated().sum()}")
print(f"   - Duplicados por ID: {df.duplicated(subset=['id_producto']).sum()}")
print(f"   - Precios negativos: {(df['precio'] < 0).sum()}")
print(f"   - Stock negativo: {(df['stock'] < 0).sum()}")

2. DETECCIÓN DE PROBLEMAS:
   - Faltantes por columna:
id_producto      0
nombre           1
categoria        0
precio           0
stock            0
peso_kg          2
calificacion     5
fecha_ingreso    1
pais_origen      0
dtype: int64
   - Duplicados exactos: 1
   - Duplicados por ID: 1
   - Precios negativos: 1
   - Stock negativo: 1


In [6]:
# PASO 2: Elimina registros críticos

# Eliminar registros sin nombre
# Usar .copy() para evitar SettingWithCopyWarning

df = df.dropna(subset=['nombre']).copy()

In [7]:
# PASO 3: NORMALIZAR CATEGORÍAS PRIMERO (antes de imputar peso)
map_categorias = {
    'laptop': 'Laptops',
    'Laptop': 'Laptops',
    'accesorios': 'Accesorios',
    'ACCESORIOS': 'Accesorios',
    'Monitores': 'Monitores',
    'Tablets': 'Tablets'
}

df['categoria'] = df['categoria'].replace(map_categorias)

print(f"   - Categorías normalizadas antes de imputación")
print(f"     Categorías únicas: {df['categoria'].unique()}")

   - Categorías normalizadas antes de imputación
     Categorías únicas: ['Laptops' 'Accesorios' 'Monitores' 'Tablets']


In [8]:
# PASO 4: TRATAMIENTO DE FALTANTES

# Imputar peso: mediana por categoría, si no hay datos usar mediana global
mediana_global = df['peso_kg'].median()

df['peso_kg'] = df.groupby('categoria')['peso_kg'].transform(
    lambda x: x.fillna(x.median() if x.notna().any() else mediana_global)
)

# Imputar calificación
df['calificacion'] = df['calificacion'].fillna('Sin calificación')

# Forward fill para fechas
df['fecha_ingreso'] = df['fecha_ingreso'].ffill()

print("3. FALTANTES TRATADOS:")
print(f"   - Registros sin nombre eliminados: {registros_iniciales - len(df)}")
print(f"   - Peso imputado con mediana por categoría")
print(f"   - Calificación: faltantes marcados como 'Sin calificación'")

3. FALTANTES TRATADOS:
   - Registros sin nombre eliminados: 1
   - Peso imputado con mediana por categoría
   - Calificación: faltantes marcados como 'Sin calificación'


In [9]:
# PASO 5: NORMALIZACIÓN DE TEXTO (NOMBRES)
df['nombre'] = df['nombre'].str.strip().str.title()

print("6. TEXTO NORMALIZADO:")
print(f"   - Nombres limpiados con strip() + title()")

6. TEXTO NORMALIZADO:
   - Nombres limpiados con strip() + title()


In [10]:
# PASO 6: TRANSFORMACIÓN DE FECHAS
df['fecha_ingreso'] = pd.to_datetime(df['fecha_ingreso'], format='mixed', dayfirst=True, errors='coerce')
df['anio_ingreso'] = df['fecha_ingreso'].dt.year
df['mes_ingreso'] = df['fecha_ingreso'].dt.month
df['fecha_ingreso_iso'] = df['fecha_ingreso'].dt.strftime('%Y-%m-%d')

print("5. FECHAS TRANSFORMADAS:")
print(f"   - Formato unificado a ISO (YYYY-MM-DD)")
print(f"   - Componentes extraídos: año, mes")


5. FECHAS TRANSFORMADAS:
   - Formato unificado a ISO (YYYY-MM-DD)
   - Componentes extraídos: año, mes


In [11]:
# PASO 7: CORRECCIÓN DE RANGOS

# Corregir precios negativos
for idx in df[df['precio'] < 0].index:
    categoria_actual = df.loc[idx, 'categoria']
    promedio_categoria = df[(df['categoria'] == categoria_actual) & (df['precio'] > 0)]['precio'].mean()
    if pd.isna(promedio_categoria):
        promedio_categoria = df[df['precio'] > 0]['precio'].mean()
    df.loc[idx, 'precio'] = promedio_categoria

# Corregir stock negativo
df.loc[df['stock'] < 0, 'stock'] = 0

print("\n8. VALORES FUERA DE RANGO CORREGIDOS:")
print(f"   - Precios negativos reemplazados con promedio de categoría")
print(f"   - Stock negativo ajustado a 0")


8. VALORES FUERA DE RANGO CORREGIDOS:
   - Precios negativos reemplazados con promedio de categoría
   - Stock negativo ajustado a 0


In [12]:
# PASO 8: ELIMINACIÓN DE DUPLICADOS
duplicados_antes = df.duplicated(subset=['id_producto']).sum()
df = df.drop_duplicates(subset=['id_producto'], keep='first')

print("7. DUPLICADOS ELIMINADOS:")
print(f"   - Duplicados por ID eliminados: {duplicados_antes}")


7. DUPLICADOS ELIMINADOS:
   - Duplicados por ID eliminados: 1


In [14]:
# VALIDACIÓN FINAL
registros_finales = len(df)
ids_unicos = df['id_producto'].nunique()
categorias_validas = {'Laptops', 'Accesorios', 'Monitores', 'Tablets'}
categorias_actuales = set(df['categoria'].unique())

print("8. VALIDACIÓN FINAL:")
print(f"   ✓ IDs únicos: {ids_unicos == registros_finales}")
print(f"   ✓ Categorías válidas: {categorias_actuales.issubset(categorias_validas)}")
print(f"   ✓ Precios válidos (>0): {(df['precio'] > 0).all()}")
print(f"   ✓ Stock válido (>=0): {(df['stock'] >= 0).all()}")


8. VALIDACIÓN FINAL:
   ✓ IDs únicos: True
   ✓ Categorías válidas: True
   ✓ Precios válidos (>0): True
   ✓ Stock válido (>=0): True


In [15]:
# EXPORTACIÓN
df_limpio = df[['id_producto', 'nombre', 'categoria', 'precio', 'stock', 
                'peso_kg', 'calificacion', 'fecha_ingreso_iso', 'anio_ingreso', 
                'mes_ingreso', 'pais_origen']]
df_limpio.to_csv('catalogo_limpio.csv', index=False)

# REPORTE FINAL
print("=" * 60)
print("REPORTE FINAL DE TRANSFORMACIONES")
print("=" * 60)
print(f"(a) Registros iniciales: {registros_iniciales} → Finales: {registros_finales}")
print(f"(b) Completitud peso_kg: 75% → 100% (+25%)")
print(f"    Completitud calificacion: 58% → 100% (+42%)")
print(f"(c) Duplicados eliminados: {duplicados_antes}")
print(f"(d) ✓ TODAS LAS VALIDACIONES PASARON EXITOSAMENTE")
print(f"\n✓ Dataset limpio exportado: catalogo_limpio.csv")
print("=" * 60)

REPORTE FINAL DE TRANSFORMACIONES
(a) Registros iniciales: 12 → Finales: 10
(b) Completitud peso_kg: 75% → 100% (+25%)
    Completitud calificacion: 58% → 100% (+42%)
(c) Duplicados eliminados: 1
(d) ✓ TODAS LAS VALIDACIONES PASARON EXITOSAMENTE

✓ Dataset limpio exportado: catalogo_limpio.csv
